# Compare policy performance

Reads a simulation log (one row per epoch x day x product, written by the policy notebooks)
and summarises each epoch and product.

In [1]:
import polars as pl
from pathlib import Path

# Relative to this notebook (code/sim/) -> results/ at the repo root.
RESULTS = Path('../../results')
LOG_FILE = RESULTS / 'Newsvendor_SingleSupplier_Pricing_PPO.parquet'

sim_log = pl.read_parquet(LOG_FILE)
sim_log.head()

epoch,seed,day,product,demand,sold,price,wastage,order_qty,order_cost,stock
i32,i64,i32,str,i64,i64,f64,i64,i64,f64,i64
0,0,0,"""FOODS_1""",1305,703,2.5,0,null,null,69918
0,0,0,"""FOODS_2""",1073,602,3.3,0,null,null,95974
0,0,0,"""FOODS_3""",7602,3590,2.3,0,null,null,55779
0,0,0,"""HOBBIES_1""",1798,894,3.7,0,null,null,668726
0,0,0,"""HOBBIES_2""",36,14,2.3,0,null,null,34535


## Performance per epoch and product

* **revenue** = units sold x the price they sold at
* **stockout_days** = days that end with no stock on hand
* **lost_units** on a stockout day = expected buyers - units sold. Expected buyers = demand x
  the share of customers who bought at that price on days the product was in stock (same epoch,
  product and price; falls back to the epoch-product share if that price was never in stock).
* **underage_cost** = lost_units x unit margin, where unit margin = price - average unit cost of
  that epoch's orders (floored at 0).

In [2]:
def performance(log):
    """Totals per (epoch, product) from a daily simulation log."""
    log = log.with_columns(
        (pl.col('stock') == 0).alias('stockout'),
        (pl.col('sold') * pl.col('price')).alias('revenue'),
    )

    # Share of demand that buys at a price, measured on in-stock days (demand there is not
    # capped by stock). Per epoch/product/price, with an epoch/product fallback.
    in_stock = log.filter(~pl.col('stockout') & (pl.col('demand') > 0))
    share_at_price = in_stock.group_by('epoch', 'product', 'price').agg(
        (pl.col('sold').sum() / pl.col('demand').sum()).alias('share_price')
    )
    share_overall = in_stock.group_by('epoch', 'product').agg(
        (pl.col('sold').sum() / pl.col('demand').sum()).alias('share_overall')
    )

    # Average unit cost of everything ordered in the epoch (null if nothing was ordered).
    unit_cost = log.group_by('epoch', 'product').agg(
        (pl.col('order_cost').sum() / pl.col('order_qty').sum()).alias('unit_cost')
    )

    shortfall = (
        log.filter(pl.col('stockout'))
        .join(share_at_price, on=['epoch', 'product', 'price'], how='left')
        .join(share_overall, on=['epoch', 'product'], how='left')
        .join(unit_cost, on=['epoch', 'product'], how='left')
        .with_columns(
            (pl.col('demand') * pl.coalesce('share_price', 'share_overall') - pl.col('sold'))
            .clip(lower_bound=0).alias('lost_units'),
            (pl.col('price') - pl.col('unit_cost').fill_null(0)).clip(lower_bound=0).alias('unit_margin'),
        )
        .group_by('epoch', 'product')
        .agg(
            pl.col('lost_units').sum(),
            (pl.col('lost_units') * pl.col('unit_margin')).sum().alias('underage_cost'),
        )
    )

    return (
        log.group_by('epoch', 'product')
        .agg(
            pl.col('demand').sum().alias('total_demand'),
            pl.col('sold').sum().alias('total_sold'),
            pl.col('revenue').sum().alias('total_revenue'),
            pl.col('wastage').sum().alias('total_wastage'),
            pl.col('order_qty').sum().alias('total_order_qty'),      # nulls (no order) skipped
            pl.col('order_cost').sum().alias('total_cost'),
            pl.col('stockout').sum().alias('stockout_days'),
        )
        .join(shortfall, on=['epoch', 'product'], how='left')
        .with_columns(pl.col('lost_units', 'underage_cost').fill_null(0))  # never out of stock
        .sort('epoch', 'product')
    )


perf = performance(sim_log)
perf

epoch,product,total_demand,total_sold,total_revenue,total_wastage,total_order_qty,total_cost,stockout_days,lost_units,underage_cost
i32,str,i64,i64,f64,i64,i64,f64,u32,f64,f64
0,"""FOODS_1""",2541283,2489185,4.9041e6,3055,2455500,769780.0,40,49584.404587,82136.954917
0,"""FOODS_2""",2002499,1923628,4.8875e6,813,1887000,938910.0,47,67567.08295,148049.60107
0,"""FOODS_3""",12475477,12297919,2.1674e7,272515,12561780,4701899.6,39,169233.679231,240913.805704
0,"""HOBBIES_1""",2763083,2036328,6.1093e6,0,1423000,670460.0,0,0.0,0.0
0,"""HOBBIES_2""",227900,225208,238269.58,0,200000,58700.0,13,1077.0,394.7205
…,…,…,…,…,…,…,…,…,…,…
99,"""FOODS_3""",12543597,12367531,2.1798e7,240607,12603820,4.7308e6,39,167532.520668,238114.377007
99,"""HOBBIES_1""",2768660,1960454,6.0275e6,0,1367500,638725.0,0,0.0,0.0
99,"""HOBBIES_2""",226320,225060,227248.8,0,210000,56100.0,0,0.0,0.0


## Mean and standard deviation over epochs, per product

Each metric becomes two columns, `<metric>_mean` and `<metric>_std`, side by side. The std is
the spread across epochs (sample std, ddof=1).

In [3]:
metrics = [c for c in perf.columns if c not in ('epoch', 'product')]


def mean_std(df, by):
    """Mean and std across epochs of every metric, grouped by `by`."""
    return (
        df.group_by(by)
        .agg(
            *[pl.col(m).mean().alias(f'{m}_mean') for m in metrics],
            *[pl.col(m).std().alias(f'{m}_std') for m in metrics],
        )
        .select(*by, *[f'{m}_{s}' for m in metrics for s in ('mean', 'std')])   # mean next to std
        .sort(by[::-1])
    )


summary = mean_std(perf, ['product'])
summary

product,total_demand_mean,total_demand_std,total_sold_mean,total_sold_std,total_revenue_mean,total_revenue_std,total_wastage_mean,total_wastage_std,total_order_qty_mean,total_order_qty_std,total_cost_mean,total_cost_std,stockout_days_mean,stockout_days_std,lost_units_mean,lost_units_std,underage_cost_mean,underage_cost_std
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""FOODS_1""",2.5503e6,5370.751681,2.4789e6,25309.768028,4.8838e6,49858.587284,780.68,1243.047167,2.4723e6,31972.05219,783629.45,15800.103421,56.23,19.040549,69006.646086,24484.588828,114114.446503,40517.027319
"""FOODS_2""",2004525.2,6189.524446,1.9485e6,16933.033334,4.9632e6,51110.60442,895.2,1176.066007,1.87476e6,25694.443844,951423.7,23484.151099,45.95,13.839174,43555.34349,14979.572354,98496.321556,33700.675696
"""FOODS_3""",1.2497e7,27288.330069,1.2333e7,48963.107854,2.1734e7,86548.849757,224423.38,111653.89254,1.2554e7,97446.842244,4.6781e6,37693.439029,34.56,8.214695,155741.131717,44096.20547,222086.668951,63084.345632
"""HOBBIES_1""",2.7682e6,5197.752827,2.0009e6,28971.011557,6.0561e6,48527.260698,0.0,0.0,1.358675e6,49127.0963,629096.5,28550.471811,3.47,7.261988,3096.339233,5961.198476,12844.030565,25059.188152
"""HOBBIES_2""",226987.01,941.009103,222823.51,2930.907514,235045.8764,7441.488534,0.0,0.0,206727.0,10097.30292,57162.01,4552.279563,16.03,17.672929,2491.97,2796.833821,967.226058,1086.392592
"""HOUSEHOLD_1""",4347166.3,14883.767804,2.3774e6,156411.570722,8.4398e6,502435.344123,0.0,0.0,1.873685e6,142918.261852,1044083.9,80780.396023,0.0,0.0,0.0,0.0,0.0,0.0
"""HOUSEHOLD_2""",1.5863e6,3372.532353,1.5597e6,16476.49024,6.8605e6,73225.506633,0.0,0.0,1.32924e6,30042.098408,967798.0,23350.332942,29.07,19.276958,22385.476704,15714.111189,96978.2512,68420.350589


## Save to Excel

Both tables, rounded to 2 decimals, in one workbook in `results/`: sheet `per_epoch` (per epoch
and product) and sheet `summary` (mean and std over epochs).

In [4]:
import polars.selectors as cs
from xlsxwriter import Workbook

excel_file = RESULTS / f'performance_{LOG_FILE.stem}.xlsx'
with Workbook(excel_file) as wb:
    perf.with_columns(cs.float().round(2)).write_excel(wb, worksheet='per_epoch')
    summary.with_columns(cs.float().round(2)).write_excel(wb, worksheet='summary')
print(f'saved {excel_file.resolve()}')

saved C:\Users\prade\OneDrive\Documents\Code\retail-revenue-management\results\performance_Newsvendor_SingleSupplier_Pricing_PPO.xlsx


## Compare every policy in results/

The policy name is taken from the file name.

In [7]:
all_perf = pl.concat([
    performance(pl.read_parquet(f)).with_columns(pl.lit(f.stem).alias('policy'))
    for f in sorted(RESULTS.glob('*.parquet'))
])

comparison = mean_std(all_perf, ['policy', 'product'])   # sorted by product, then policy

comparison_excel_file = RESULTS / 'model_comparison.xlsx'
comparison.write_excel(comparison_excel_file)

comparison

policy,product,total_demand_mean,total_demand_std,total_sold_mean,total_sold_std,total_revenue_mean,total_revenue_std,total_wastage_mean,total_wastage_std,total_order_qty_mean,total_order_qty_std,total_cost_mean,total_cost_std,stockout_days_mean,stockout_days_std,lost_units_mean,lost_units_std,underage_cost_mean,underage_cost_std
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Newsvendor_SingleSupplier_DP_1…","""FOODS_1""",2.5503e6,5370.751681,1611601.3,53903.569416,3.9181e6,90613.920433,5517.21,3288.37713,1.609065e6,59598.388859,507744.75,19593.369864,7.38,8.080129,5364.913859,5714.187337,11301.749023,12093.902355
"""Newsvendor_SingleSupplier_DP_D…","""FOODS_1""",2.5503e6,5370.751681,1.5424e6,20594.912532,3.7964e6,38126.672188,21376.91,4219.396889,1.559475e6,28451.924275,491137.1,11502.740191,13.22,9.329631,8025.458103,6242.644817,17283.787906,13411.543919
"""Newsvendor_SingleSupplier_DP_S…","""FOODS_1""",2.5503e6,5370.751681,1611601.3,53903.569416,3.9181e6,90613.920433,5517.21,3288.37713,1.609065e6,59598.388859,507744.75,19593.369864,7.38,8.080129,5364.913859,5714.187337,11301.749023,12093.902355
"""Newsvendor_SingleSupplier_Pric…","""FOODS_1""",2.5503e6,5370.751681,2.4789e6,25309.768028,4.8838e6,49858.587284,780.68,1243.047167,2.4723e6,31972.05219,783629.45,15800.103421,56.23,19.040549,69006.646086,24484.588828,114114.446503,40517.027319
"""Newswendor_SingleSupplier_Cons…","""FOODS_1""",2.5503e6,5370.751681,1.3689e6,4923.159461,3422266.6,12307.898653,28516.93,4469.589447,1.38936e6,21303.96653,436835.35,10104.532968,9.52,6.756187,5025.001861,4436.604855,10981.982739,9702.751428
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Newsvendor_SingleSupplier_DP_D…","""HOUSEHOLD_2""",1.5863e6,3372.532353,934011.47,13895.909561,4.7642e6,68535.706845,0.0,0.0,705700.0,27895.005165,514105.2,20392.256328,3.9,6.679533,2040.310435,3525.822328,8944.231327,15455.516008
"""Newsvendor_SingleSupplier_DP_S…","""HOUSEHOLD_2""",1.5863e6,3372.532353,1.0907e6,11327.767275,5.4620e6,45313.623915,0.0,0.0,861040.0,23902.968836,626949.4,18730.618687,5.36,8.586482,3061.327639,5082.616943,13113.331858,21727.40173
"""Newsvendor_SingleSupplier_Pric…","""HOUSEHOLD_2""",1.5863e6,3372.532353,1.5597e6,16476.49024,6.8605e6,73225.506633,0.0,0.0,1.32924e6,30042.098408,967798.0,23350.332942,29.07,19.276958,22385.476704,15714.111189,96978.2512,68420.350589
